# 노트북 10. 추론 모델 심화 — Thinking, Budget, 파라미터 전략

> Phase 3 — 실전 기법: 챗봇을 똑똑하게 만드는 기술

모든 질문에 '깊은 생각'이 필요하지는 않습니다.
추론 모델의 작동 원리와 다양한 제어 인자를 이해해야 비용-정확도 최적점을 찾을 수 있습니다.

**학습 목표**
- 추론 모델(Thinking Model)의 작동 원리를 이해한다
- thinking_budget으로 추론 깊이를 제어할 수 있다
- thinking_tokens의 과금 구조를 이해하고 비용을 계산할 수 있다
- include_thoughts로 모델의 추론 과정을 확인할 수 있다
- 질문 유형에 따라 적절한 thinking_budget을 선택할 수 있다

**구성**
| Part | 내용 | 형태 |
|------|------|------|
| Part 1 — 이론 | 추론 모델 원리 + budget 제어 + 비용 구조 | 읽고 실행 |
| Part 2 — 실습 | budget 비교 + 추론 과정 확인 + 비용 모니터링 | 직접 작성 |
| Part 3 — 챌린지 | 질문 유형별 최적 budget 탐색 | 강사와 함께 |

In [ ]:
# 환경 설정 — 이 셀을 먼저 실행하세요
!pip install -q google-genai langchain-google-genai

import os
import json
import time
from google import genai
from google.genai import types

# API 키 설정 (Colab 환경 기준)
from google.colab import userdata
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)
MODEL = "gemini-2.5-flash"

print("환경 설정 완료")

---

# Part 1 — 이론

개념을 마크다운 설명과 실행 가능한 코드 예시로 배웁니다.
읽고 실행하면서 따라와 주세요.

## 1.1 추론 모델이란?

**일반 LLM**과 **추론 모델**의 차이를 이해하는 것이 출발점입니다.

| 구분 | 일반 LLM | 추론 모델 |
|------|---------|----------|
| 동작 방식 | 프롬프트 → 바로 답변 생성 | 프롬프트 → 내부 추론(CoT) → 답변 생성 |
| 토큰 흐름 | input → output | input → thinking → output |
| 강점 | 빠른 응답, 낮은 비용 | 복잡한 문제 해결, 높은 정확도 |
| 약점 | 복잡한 논리에 약함 | 느림, 비용 높음 |

추론 모델은 답변 전에 문제를 분해하고, 가설을 세우고, 검증하는 과정을 자동으로 수행합니다.
이 과정을 **Chain-of-Thought(CoT)**라고 합니다.

### 핵심 차이: 추가 토큰을 소비하는 구조

```
일반 LLM:     [input tokens] → [output tokens]
추론 모델:    [input tokens] → [thinking tokens] → [output tokens]
                                    ↑
                         사용자에게 보이지 않지만
                         비용은 발생함
```

thinking tokens은 모델이 '생각하는 데' 사용하는 토큰입니다.
답변 품질을 위해 **추가 비용(시간+돈)을 지불**하는 구조입니다.

### 언제 추론 모델을 사용해야 하는가?

| 추론 필요 | 추론 불필요 |
|-----------|------------|
| 수학 문제 풀이 | 인사, 일상 대화 |
| 코드 버그 분석 | 단순 번역 |
| 논리 퍼즐 | 텍스트 요약 (단순) |
| 다단계 추론 | 분류 (감성, 카테고리) |
| 전략 수립 | 정보 조회 |

**원칙**: 사람이 "잠깐 생각해봐야" 하는 질문에 추론 모델을 사용합니다.

In [ ]:
# 추론 비활성화 (thinking_budget=0) — 일반 LLM처럼 동작
question = "12345 * 67890의 결과는?"

start = time.time()
response_no_think = client.models.generate_content(
    model=MODEL,
    contents=question,
    config={"thinking_config": {"thinking_budget": 0}},
)
time_no_think = time.time() - start

print("=== thinking_budget=0 (추론 비활성화) ===")
print(f"답변: {response_no_think.text[:100]}")
print(f"시간: {time_no_think:.2f}초")
usage = response_no_think.usage_metadata
print(f"input: {usage.prompt_token_count}, output: {usage.candidates_token_count}")
thinking_tokens = getattr(usage, 'thoughts_token_count', 0) or 0
print(f"thinking: {thinking_tokens}")

In [ ]:
# 추론 활성화 (thinking_budget=4096) — 생각한 후 답변
start = time.time()
response_think = client.models.generate_content(
    model=MODEL,
    contents=question,
    config={"thinking_config": {"thinking_budget": 4096}},
)
time_think = time.time() - start

print("=== thinking_budget=4096 (추론 활성화) ===")
print(f"답변: {response_think.text[:100]}")
print(f"시간: {time_think:.2f}초")
usage = response_think.usage_metadata
print(f"input: {usage.prompt_token_count}, output: {usage.candidates_token_count}")
thinking_tokens = getattr(usage, 'thoughts_token_count', 0) or 0
print(f"thinking: {thinking_tokens}")

print(f"\n시간 차이: {time_think - time_no_think:+.2f}초")

> **실행 결과 해석**
> - `thinking_budget=0`: thinking_tokens가 0이고, 응답이 빠릅니다
> - `thinking_budget=4096`: thinking_tokens가 0보다 크고, 응답에 시간이 더 걸립니다
> - 같은 질문이어도 추론 여부에 따라 답변의 정확도와 상세함이 달라질 수 있습니다

> 같은 모델(`gemini-2.5-flash`)이 `thinking_budget` 설정만으로 추론/비추론을 전환합니다.
> 별도의 추론 전용 모델을 사용할 필요가 없습니다.

In [ ]:
# 지원되지 않는 모델에서 thinking 설정 시 동작 확인
try:
    resp_unsupported = client.models.generate_content(
        model="gemini-2.0-flash",
        contents="안녕하세요",
        config={"thinking_config": {"thinking_budget": 1024}},
    )
    print(f"gemini-2.0-flash: 응답은 받았지만 thinking은 무시될 수 있음")
    print(f"  text: {resp_unsupported.text[:50]}")
    usage = resp_unsupported.usage_metadata
    thinking = getattr(usage, 'thoughts_token_count', 0) or 0
    print(f"  thinking_tokens: {thinking} (0이면 무시됨)")
except Exception as e:
    print(f"gemini-2.0-flash 에러: {type(e).__name__}: {str(e)[:100]}")
    print("→ thinking을 지원하지 않는 모델에서는 에러가 발생할 수 있습니다")

## 1.2 Gemini 세대별 추론 제어

| 세대 | 제어 방식 | 값 범위 | 특징 |
|------|----------|---------|------|
| Gemini 2.0 (Flash 등) | 추론 미지원 | - | thinking 파라미터 무시됨 |
| Gemini 2.5 (Flash/Pro) | `thinking_budget` | 0 ~ 24576 토큰 | 토큰 단위로 세밀 제어 |
| Gemini 3 계열 | `thinking_level` | low / medium / high | 단계별 간편 제어 |

**지원 모델 확인이 필수입니다.**
`gemini-2.5-flash`, `gemini-2.5-pro`만 thinking을 지원합니다.
`gemini-2.0-flash`에 thinking_budget을 설정하면 무시되거나 에러가 발생합니다.

## 1.3 thinking_budget 심화

`thinking_budget`은 모델이 추론에 사용할 수 있는 **최대 토큰 수**입니다.
실제 사용량은 이보다 적을 수 있습니다 (모델이 일찍 추론을 끝낼 수 있음).

| 범위 | 용도 | 예시 |
|------|------|------|
| `0` | 추론 비활성화 | 인사, 번역, 단순 요약 |
| `128 ~ 1024` | 가벼운 추론 | 감성 분류, 짧은 논리 판단 |
| `1024 ~ 4096` | 중간 수준 | 일반 분석, 코드 리뷰, 문서 요약 판단 |
| `4096 ~ 8192` | 깊은 추론 | 수학 증명, 복잡한 코드 디버깅 |
| `8192 ~ 24576` | 최대 추론 | 연구 수준의 복잡한 문제 |

> **과잉 추론(Overthinking) 주의**
>
> budget을 높인다고 항상 정확도가 올라가는 것은 아닙니다.
> 단순한 문제에 높은 budget을 설정하면 모델이 불필요하게 깊이 생각하여
> 오히려 잘못된 결론에 도달하거나, 응답이 지나치게 느려질 수 있습니다.
> 질문의 복잡도에 맞는 적절한 budget을 선택하는 것이 핵심입니다.

In [ ]:
# 과잉 추론 데모: 단순한 질문에 높은 budget
simple_question = "1 + 1은?"

for budget in [0, 512, 8192]:
    start = time.time()
    resp = client.models.generate_content(
        model=MODEL, contents=simple_question,
        config={"thinking_config": {"thinking_budget": budget}},
    )
    elapsed = time.time() - start
    usage = resp.usage_metadata
    thinking = getattr(usage, 'thoughts_token_count', 0) or 0
    print(f"budget={budget:>5}: time={elapsed:.1f}s, thinking={thinking:>4}, "
          f"답변={resp.text[:30]}")

print("\n→ 단순한 질문에 높은 budget은 시간/비용만 낭비됩니다")

In [ ]:
# thinking_budget 값별 비교 실험
budgets = [0, 512, 2048, 8192]
question = "3의 15승을 단계별로 계산해줘"

print(f"질문: {question}\n")
for budget in budgets:
    start = time.time()
    resp = client.models.generate_content(
        model=MODEL,
        contents=question,
        config={"thinking_config": {"thinking_budget": budget}},
    )
    elapsed = time.time() - start
    usage = resp.usage_metadata
    thinking = getattr(usage, 'thoughts_token_count', 0) or 0
    print(f"budget={budget:>5}: "
          f"time={elapsed:.1f}s, "
          f"thinking={thinking:>4}, "
          f"output={usage.candidates_token_count:>4}, "
          f"답변={resp.text[:50]}")

> 위 결과에서 budget이 높아질수록 thinking_tokens가 증가하지만,
> 실제 사용량이 budget에 도달하지 않을 수 있습니다.
> 이는 모델이 "충분히 추론했다"고 판단하면 일찍 종료하기 때문입니다.
> budget은 **상한값**일 뿐, **목표값**이 아닙니다.

## 1.4 Thinking Token과 과금 구조

추론 모델의 토큰은 **3종류**로 분리됩니다:

| 토큰 종류 | 설명 | 과금 |
|-----------|------|------|
| `input_tokens` | 프롬프트 (사용자 메시지 + system prompt + 대화 이력) | input 단가 |
| `output_tokens` | 최종 답변 텍스트 | output 단가 |
| `thinking_tokens` | 내부 추론 과정 (사용자에게 보이지 않음) | **output 단가** |

thinking_tokens도 output_tokens 단가로 과금됩니다.
추론이 깊을수록 비용이 증가합니다.

In [ ]:
# 토큰 사용량 상세 확인
response = client.models.generate_content(
    model=MODEL,
    contents="대한민국의 GDP가 세계에서 몇 번째인지 분석해줘",
    config={"thinking_config": {"thinking_budget": 4096}},
)

usage = response.usage_metadata
input_tokens = usage.prompt_token_count
output_tokens = usage.candidates_token_count
thinking_tokens = getattr(usage, 'thoughts_token_count', 0) or 0

print("=== 토큰 사용량 ===")
print(f"  input_tokens:    {input_tokens:>6}")
print(f"  output_tokens:   {output_tokens:>6}")
print(f"  thinking_tokens: {thinking_tokens:>6}")
print(f"  합계:            {input_tokens + output_tokens + thinking_tokens:>6}")

# 비용 추정 (gemini-2.5-flash 기준 예시 단가)
INPUT_PRICE = 0.15 / 1_000_000   # $0.15 / 1M tokens
OUTPUT_PRICE = 0.60 / 1_000_000  # $0.60 / 1M tokens

cost_input = input_tokens * INPUT_PRICE
cost_output = output_tokens * OUTPUT_PRICE
cost_thinking = thinking_tokens * OUTPUT_PRICE  # thinking도 output 단가
total_cost = cost_input + cost_output + cost_thinking

print(f"\n=== 비용 추정 (USD) ===")
print(f"  input:    ${cost_input:.6f}")
print(f"  output:   ${cost_output:.6f}")
print(f"  thinking: ${cost_thinking:.6f}")
print(f"  합계:     ${total_cost:.6f}")
print(f"  thinking 비중: {cost_thinking/total_cost*100:.1f}%" if total_cost > 0 else "")

> thinking_tokens이 전체 비용의 상당 부분을 차지할 수 있습니다.
> budget=0과 budget=4096의 비용 차이를 항상 인식해야 합니다.

### 비용 계산 공식

```
총 비용 = (input_tokens x input_단가) + (output_tokens x output_단가) + (thinking_tokens x output_단가)
```

thinking_tokens는 output 단가로 과금되므로,
`gemini-2.5-pro`에서 높은 budget은 비용이 급격히 증가합니다.

In [ ]:
# budget=0 vs budget=4096 비용 비교
question = "인공지능의 미래에 대해 간단히 설명해줘"

resp_0 = client.models.generate_content(
    model=MODEL, contents=question,
    config={"thinking_config": {"thinking_budget": 0}},
)
resp_4096 = client.models.generate_content(
    model=MODEL, contents=question,
    config={"thinking_config": {"thinking_budget": 4096}},
)

def calc_cost(usage):
    inp = usage.prompt_token_count * INPUT_PRICE
    out = usage.candidates_token_count * OUTPUT_PRICE
    think = (getattr(usage, 'thoughts_token_count', 0) or 0) * OUTPUT_PRICE
    return inp + out + think

cost_0 = calc_cost(resp_0.usage_metadata)
cost_4096 = calc_cost(resp_4096.usage_metadata)

print(f"budget=0:    ${cost_0:.6f}")
print(f"budget=4096: ${cost_4096:.6f}")
print(f"비용 배수:   {cost_4096/cost_0:.1f}x" if cost_0 > 0 else "")

## 1.5 추론 과정 확인: include_thoughts

`include_thoughts=True`를 설정하면 모델의 내부 추론 과정을 응답에 포함합니다.
응답의 content가 리스트 형태로 바뀝니다:

```python
[
    {"type": "text", "text": "추론 과정...", "thought": True},  # thinking
    {"type": "text", "text": "최종 답변..."}                     # response
]
```

디버깅, 교육, 품질 검증에 유용합니다: "모델이 왜 이렇게 답했는지" 직접 확인할 수 있습니다.

In [ ]:
# include_thoughts=True로 추론 과정 확인
response = client.models.generate_content(
    model=MODEL,
    contents="3개의 상자에 각각 빨강, 파랑, 초록 공이 하나씩 있다. "
             "빨강 공과 파랑 공의 상자를 바꾸고, 다시 파랑 공과 초록 공의 상자를 바꾸면 "
             "각 상자에 어떤 색 공이 있을까?",
    config={"thinking_config": {
        "thinking_budget": 4096,
        "include_thoughts": True,
    }},
)

# content 구조 확인
parts = response.candidates[0].content.parts
print(f"parts 수: {len(parts)}\n")

for i, part in enumerate(parts):
    is_thought = getattr(part, 'thought', False)
    label = "[THOUGHT]" if is_thought else "[RESPONSE]"
    text = part.text[:200] if part.text else "(없음)"
    print(f"{label} (part {i})")
    print(f"  {text}")
    print()

In [ ]:
# thought와 response를 분리하는 유틸리티 함수
def extract_thought_and_response(response):
    """응답에서 thought와 response를 분리합니다."""
    parts = response.candidates[0].content.parts
    thought_parts = []
    response_parts = []

    for part in parts:
        if getattr(part, 'thought', False):
            thought_parts.append(part.text)
        else:
            response_parts.append(part.text)

    return "\n".join(thought_parts), "\n".join(response_parts)

thought, answer = extract_thought_and_response(response)
print(f"=== 추론 과정 (일부) ===")
print(thought[:300])
print(f"\n=== 최종 답변 ===")
print(answer[:300])

> `extract_thought_and_response()` 함수는 이 노트북 전반에서 재사용됩니다.
> 실습에서도 이 함수를 활용하여 추론 과정과 답변을 분리합니다.

> **include_thoughts 주의사항**
> - `include_thoughts`는 응답 포맷만 바꿉니다. 추론 자체를 활성화하지 않습니다.
> - `thinking_budget=0`이면 thought 블록이 비어 있습니다.
> - 추론 과정을 보려면 `thinking_budget > 0`과 `include_thoughts=True`를 함께 설정해야 합니다.

In [ ]:
# budget=0 + include_thoughts=True → thought 블록이 비어 있음
resp_no_budget = client.models.generate_content(
    model=MODEL,
    contents="1 + 1은?",
    config={"thinking_config": {
        "thinking_budget": 0,
        "include_thoughts": True,
    }},
)

parts = resp_no_budget.candidates[0].content.parts
thought_parts = [p for p in parts if getattr(p, 'thought', False)]
response_parts = [p for p in parts if not getattr(p, 'thought', False)]

print(f"thought parts: {len(thought_parts)} (budget=0이므로 비어 있음)")
print(f"response parts: {len(response_parts)}")
if response_parts:
    print(f"답변: {response_parts[0].text[:50]}")

## 1.6 Thinking과 다른 파라미터의 관계

Thinking 모드에서는 일부 파라미터에 **제약**이 있습니다.

| 파라미터 | thinking_budget=0 | thinking_budget > 0 |
|---------|-------------------|--------------------|
| temperature | 자유 설정 (0.0~2.0) | 고정: 1.0 (변경 불가) |
| top_p | 자유 설정 | 일부 제약 가능 |
| top_k | 자유 설정 | 일부 제약 가능 |
| response_mime_type | 사용 가능 | 사용 가능 |

**temperature 제약**이 가장 중요합니다.
thinking 모드에서 temperature를 0으로 설정하면 에러가 발생합니다.

In [ ]:
# thinking 모드에서 temperature 제약 확인
try:
    # thinking 활성화 + temperature=0 → 에러 예상
    resp = client.models.generate_content(
        model=MODEL,
        contents="안녕하세요",
        config={
            "thinking_config": {"thinking_budget": 1024},
            "temperature": 0.0,  # thinking 모드에서 허용되지 않음
        },
    )
    print(f"성공: {resp.text[:50]}")
except Exception as e:
    print(f"에러 발생: {type(e).__name__}")
    print(f"  {str(e)[:200]}")
    print("\n→ thinking 모드에서는 temperature를 1.0으로 고정해야 합니다")

In [ ]:
# thinking 모드에서 올바른 설정
resp = client.models.generate_content(
    model=MODEL,
    contents="간단한 논리 퍼즐: A > B, B > C이면 A와 C의 관계는?",
    config={
        "thinking_config": {"thinking_budget": 1024},
        "temperature": 1.0,  # thinking 모드에서는 1.0 고정
    },
)
print(f"답변: {resp.text[:200]}")

> **Thinking 모드 파라미터 요약**
> - `thinking_budget > 0`이면 temperature는 반드시 1.0 (명시하지 않으면 자동 적용)
> - `thinking_budget = 0`이면 temperature를 자유롭게 설정 가능
> - top_p, top_k도 thinking 모드에서 제약이 있을 수 있으므로 주의
> - response_mime_type(Structured Output)은 thinking 모드와 함께 사용 가능

## 1.7 LangChain에서 Thinking 사용

LangChain의 `ChatGoogleGenerativeAI`에서도 thinking을 제어할 수 있습니다.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# thinking 비활성화
llm_no_think = ChatGoogleGenerativeAI(
    model=MODEL,
    google_api_key=GEMINI_API_KEY,
    thinking_budget=0,
)

# thinking 활성화
llm_think = ChatGoogleGenerativeAI(
    model=MODEL,
    google_api_key=GEMINI_API_KEY,
    thinking_budget=4096,
)

question = "파이썬에서 리스트와 튜플의 주요 차이점 3가지를 알려줘"

result_no = llm_no_think.invoke(question)
result_yes = llm_think.invoke(question)

print("=== thinking_budget=0 ===")
print(f"  {result_no.content[:100]}")
print(f"  tokens: {result_no.usage_metadata}")

print("\n=== thinking_budget=4096 ===")
print(f"  {result_yes.content[:100]}")
print(f"  tokens: {result_yes.usage_metadata}")

In [ ]:
# LangChain에서 thinking 응답의 content 구조 확인
# LangChain은 thinking 과정을 AIMessage.content에 포함할 수 있음
llm_think_lc = ChatGoogleGenerativeAI(
    model=MODEL,
    google_api_key=GEMINI_API_KEY,
    thinking_budget=2048,
)

result = llm_think_lc.invoke("5! (5 팩토리얼)은?")
print(f"content type: {type(result.content).__name__}")
if isinstance(result.content, list):
    print(f"content 항목 수: {len(result.content)}")
    for i, item in enumerate(result.content):
        if isinstance(item, dict):
            is_thought = item.get("thought", False)
            text = item.get("text", "")[:100]
            print(f"  [{i}] thought={is_thought}: {text}")
        else:
            print(f"  [{i}] {str(item)[:100]}")
else:
    print(f"content: {str(result.content)[:150]}")

## 1.8 질문 유형별 최적 budget 전략

모든 질문에 높은 budget을 쓰면 비용이 낭비됩니다.
질문의 복잡도에 따라 budget을 조절하는 것이 효율적입니다.

| 질문 유형 | 권장 budget | 이유 |
|-----------|------------|------|
| 인사, 일상 대화 | 0 | 추론 불필요 |
| 번역, 단순 요약 | 0 ~ 512 | 패턴 매칭으로 충분 |
| 분류, 감성 분석 | 512 ~ 1024 | 가벼운 판단 |
| 코드 리뷰, 분석 | 1024 ~ 4096 | 논리적 분석 필요 |
| 수학, 논리 퍼즐 | 4096 ~ 8192 | 단계적 추론 필요 |
| 복잡한 디버깅 | 8192+ | 깊은 분석 |

> **budget 선택 경험 법칙**
> 1. 정답이 확실한 단순 질문 → `0`
> 2. "설명해줘" 류의 중간 복잡도 → `1024 ~ 2048`
> 3. "왜?" "증명해줘" 류의 깊은 분석 → `4096 ~ 8192`
> 4. 확실하지 않으면 `1024`에서 시작하여 품질을 확인하고 조절

In [ ]:
# 질문 유형별 budget 비교 실험
test_cases = [
    ("안녕하세요!", 0, "인사"),
    ("'사과'를 영어로 번역해줘", 0, "번역"),
    ("이 리뷰는 긍정인가 부정인가? '배송이 빠르고 품질이 좋습니다'", 512, "분류"),
    ("파이썬의 GIL이 멀티스레딩 성능에 미치는 영향을 설명해줘", 2048, "기술 분석"),
    ("1부터 100까지의 소수를 모두 나열하고 개수를 알려줘", 4096, "수학"),
]

print(f"{'유형':<10} {'budget':>6} {'thinking':>8} {'output':>7} {'time':>6}")
print("-" * 45)

for question, budget, label in test_cases:
    start = time.time()
    resp = client.models.generate_content(
        model=MODEL, contents=question,
        config={"thinking_config": {"thinking_budget": budget}},
    )
    elapsed = time.time() - start
    usage = resp.usage_metadata
    thinking = getattr(usage, 'thoughts_token_count', 0) or 0
    print(f"{label:<10} {budget:>6} {thinking:>8} {usage.candidates_token_count:>7} {elapsed:>5.1f}s")

### Thinking과 스트리밍

Thinking 모드에서도 스트리밍을 사용할 수 있습니다.
스트리밍 시 thinking 청크가 먼저 도착하고, 이후 response 청크가 도착합니다.
`include_thoughts=True`가 설정되어 있어야 thinking 청크를 받을 수 있습니다.

> **실전 전략**: 라우터를 사용하여 질문 유형을 먼저 분류하고,
> 유형에 따라 서로 다른 thinking_budget으로 모델을 호출하는 방식이 효과적입니다.
> 이 패턴은 노트북 11(ReAct 에이전트)에서 더 자세히 다룹니다.

In [ ]:
# 간단한 budget 라우터 구현 예시
def select_budget(question: str) -> int:
    """질문의 복잡도에 따라 thinking_budget을 선택합니다."""
    # 간단한 규칙 기반 라우터 (실전에서는 분류 모델 사용)
    simple_patterns = ["안녕", "번역", "요약해줘", "뭐야"]
    medium_patterns = ["설명해줘", "비교해줘", "분석해줘", "리뷰"]
    hard_patterns = ["증명", "디버깅", "풀어줘", "계산", "단계별"]

    q_lower = question.lower()

    if any(p in q_lower for p in hard_patterns):
        return 4096
    elif any(p in q_lower for p in medium_patterns):
        return 1024
    elif any(p in q_lower for p in simple_patterns):
        return 0
    else:
        return 512  # 기본값

# 테스트
test_questions = [
    "안녕하세요!",
    "파이썬의 GIL을 설명해줘",
    "이 코드의 버그를 디버깅해줘: ...",
    "피보나치 수열의 20번째 수를 단계별로 계산해줘",
]

for q in test_questions:
    budget = select_budget(q)
    print(f"budget={budget:>5}: {q[:40]}")

## 1.9 다른 모델의 추론 접근법 비교

추론 제어는 Gemini만의 기능이 아닙니다. 주요 모델별 비교표입니다.

| 모델 | 추론 제어 | 특징 |
|------|----------|------|
| Gemini 2.5 | `thinking_budget` (토큰 수) | 세밀한 토큰 단위 제어, 동일 모델에서 전환 |
| Gemini 3 | `thinking_level` (low/mid/high) | 간편한 단계 제어 |
| OpenAI o1/o3 | `reasoning_effort` (low/medium/high) | 별도 모델 계열 (o1, o3-mini 등) |
| Claude 3.5+ | `extended_thinking` + `budget_tokens` | 토큰 단위 제어, 별도 활성화 플래그 |

Gemini의 강점: 동일 모델에서 budget만 바꿔 추론/비추론 전환이 가능합니다.

> **Gemini의 추론 제어 강점**
> - 동일 모델에서 `thinking_budget=0`(비추론)과 `thinking_budget=8192`(깊은 추론)을 자유롭게 전환
> - 별도 모델을 배포하거나 API 엔드포인트를 분리할 필요 없음
> - 토큰 단위 세밀 제어로 비용 최적화 가능 (OpenAI의 low/medium/high보다 정밀)

## 1.10 Thinking과 Structured Output

Thinking 모드에서도 Structured Output을 함께 사용할 수 있습니다.
모델이 추론한 후 구조화된 JSON을 반환합니다.

In [ ]:
from pydantic import BaseModel, Field

class MathSolution(BaseModel):
    question: str = Field(description="원래 질문")
    answer: int = Field(description="최종 답 (정수)")
    steps: list[str] = Field(description="풀이 단계 (최대 5단계)")

# thinking + structured output
response = client.models.generate_content(
    model=MODEL,
    contents="연속하는 세 자연수의 합이 75일 때, 세 수를 구해줘",
    config={
        "thinking_config": {"thinking_budget": 2048},
        "response_mime_type": "application/json",
        "response_schema": MathSolution,
    },
)

result = MathSolution.model_validate_json(response.text)
print(f"질문: {result.question}")
print(f"답: {result.answer}")
for i, step in enumerate(result.steps, 1):
    print(f"  {i}. {step}")

usage = response.usage_metadata
thinking = getattr(usage, 'thoughts_token_count', 0) or 0
print(f"\nthinking_tokens: {thinking}")

> Thinking과 Structured Output을 함께 사용하면
> "정확한 추론 + 구조화된 결과"를 동시에 얻을 수 있습니다.
> 수학 문제 풀이, 코드 분석 보고서 등에 효과적입니다.

### Thinking의 실전 활용 시나리오

| 시나리오 | Thinking + Structured Output | 효과 |
|---------|----------------------------|------|
| 코드 리뷰 자동화 | 추론으로 버그 분석 → JSON 보고서 | 정확한 이슈 목록 + 자동 처리 |
| 수학 교육 앱 | 추론으로 풀이 → 단계별 JSON | 학생에게 풀이 과정 표시 |
| 법률 문서 분석 | 추론으로 쟁점 파악 → 구조화된 요약 | 변호사 검토 효율화 |
| 고객 문의 분류 | 추론으로 복합 이슈 분석 → 카테고리 분류 | 복잡한 문의도 정확히 분류 |

In [ ]:
# 토큰 사용량 모니터링 유틸리티
def monitor_thinking(response, label=""):
    """응답의 토큰 사용량을 모니터링합니다."""
    usage = response.usage_metadata
    inp = usage.prompt_token_count
    out = usage.candidates_token_count
    think = getattr(usage, 'thoughts_token_count', 0) or 0

    cost = (inp * INPUT_PRICE) + ((out + think) * OUTPUT_PRICE)

    print(f"[{label}] input={inp}, output={out}, thinking={think}, "
          f"total={inp+out+think}, cost=${cost:.6f}")
    return {"input": inp, "output": out, "thinking": think, "cost": cost}

# 테스트
resp = client.models.generate_content(
    model=MODEL,
    contents="7 + 8 * 3은?",
    config={"thinking_config": {"thinking_budget": 1024}},
)
monitor_thinking(resp, "간단한 계산")

In [ ]:
# 누적 비용 추적기
class CostTracker:
    """세션 동안의 누적 토큰 사용량과 비용을 추적합니다."""
    def __init__(self, input_price=0.15/1_000_000, output_price=0.60/1_000_000):
        self.input_price = input_price
        self.output_price = output_price
        self.total_input = 0
        self.total_output = 0
        self.total_thinking = 0
        self.call_count = 0

    def add(self, usage_metadata):
        self.total_input += usage_metadata.prompt_token_count
        self.total_output += usage_metadata.candidates_token_count
        self.total_thinking += getattr(usage_metadata, 'thoughts_token_count', 0) or 0
        self.call_count += 1

    def summary(self):
        cost = (self.total_input * self.input_price +
                (self.total_output + self.total_thinking) * self.output_price)
        print(f"호출 {self.call_count}회: "
              f"input={self.total_input}, output={self.total_output}, "
              f"thinking={self.total_thinking}, cost=${cost:.6f}")

# 사용 예시
tracker = CostTracker()

for q in ["안녕하세요", "3+5는?", "파이썬의 GIL을 설명해줘"]:
    budget = select_budget(q)
    resp = client.models.generate_content(
        model=MODEL, contents=q,
        config={"thinking_config": {"thinking_budget": budget}},
    )
    tracker.add(resp.usage_metadata)

tracker.summary()

---

### 생각해보기

1. thinking_budget을 높이면 항상 더 정확한 답변을 받을 수 있을까요? 과잉 추론(overthinking)이 발생하는 구체적인 사례를 생각해보세요.
2. thinking_tokens이 output 단가로 과금되는 이유는 무엇일까요? 만약 별도의 (더 저렴한) thinking 단가가 있다면 어떤 변화가 생길까요?
3. 추론 과정(thought)을 사용자에게 보여주는 것이 좋을까요, 감추는 것이 좋을까요? 사용 시나리오에 따라 달라질 수 있는 이유는 무엇일까요?

---

# Part 2 — 실습

이론에서 배운 개념을 직접 코드로 작성해봅니다.
TODO 부분을 채워주세요. 막히면 정답 주석을 확인하세요.

### 실습 10-1: thinking_budget=0 vs 4096 응답 비교

같은 질문에 대해 thinking_budget을 0과 4096으로 설정하여 응답을 비교하세요.

**요구사항**
1. 질문: "피보나치 수열의 10번째 수를 구해줘"
2. budget=0과 budget=4096으로 각각 호출
3. 응답 내용, 시간, 토큰 사용량을 비교하여 출력

In [ ]:
# TODO: thinking_budget=0 vs 4096 비교

# 정답
question = "피보나치 수열의 10번째 수를 구해줘"

for budget in [0, 4096]:
    start = time.time()
    resp = client.models.generate_content(
        model=MODEL, contents=question,
        config={"thinking_config": {"thinking_budget": budget}},
    )
    elapsed = time.time() - start
    usage = resp.usage_metadata
    thinking = getattr(usage, 'thoughts_token_count', 0) or 0

    print(f"=== budget={budget} ===")
    print(f"  답변: {resp.text[:100]}")
    print(f"  시간: {elapsed:.2f}초")
    print(f"  input: {usage.prompt_token_count}, output: {usage.candidates_token_count}, thinking: {thinking}")
    print()

print("TODO를 완성해주세요")

### 실습 10-2: include_thoughts로 추론 과정 확인

include_thoughts=True로 모델의 추론 과정을 확인하세요.

**요구사항**
1. 질문: "A는 B보다 키가 크고, B는 C보다 키가 크고, C는 D보다 키가 큽니다. 키 순서대로 나열해줘"
2. thinking_budget=2048, include_thoughts=True
3. thought 부분과 response 부분을 분리하여 출력
4. thought의 길이(글자 수)를 확인

In [ ]:
# TODO: include_thoughts 추론 과정 확인

# 정답
resp = client.models.generate_content(
    model=MODEL,
    contents="A는 B보다 키가 크고, B는 C보다 키가 크고, C는 D보다 키가 큽니다. 키 순서대로 나열해줘",
    config={"thinking_config": {
        "thinking_budget": 2048,
        "include_thoughts": True,
    }},
)

thought, answer = extract_thought_and_response(resp)

print("=== 추론 과정 ===")
print(thought[:300])
print(f"\n(추론 과정 길이: {len(thought)}자)")

print("\n=== 최종 답변 ===")
print(answer)

print("TODO를 완성해주세요")

### 실습 10-3: thinking_tokens 사용량 모니터링

여러 budget 값에서 실제 thinking_tokens 사용량을 측정하세요.

**요구사항**
1. 같은 질문을 budget [0, 256, 1024, 4096, 8192]로 각각 호출
2. 질문: "대한민국 역대 대통령을 재임 기간과 함께 나열해줘"
3. 각 budget에서 실제 thinking_tokens 사용량 기록
4. budget 대비 실제 사용량 비율을 계산하여 출력

In [ ]:
# TODO: thinking_tokens 사용량 모니터링

# 정답
question = "대한민국 역대 대통령을 재임 기간과 함께 나열해줘"
budgets = [0, 256, 1024, 4096, 8192]

print(f"{'budget':>6} {'실제 thinking':>12} {'output':>7} {'비율':>6}")
print("-" * 35)

for budget in budgets:
    resp = client.models.generate_content(
        model=MODEL, contents=question,
        config={"thinking_config": {"thinking_budget": budget}},
    )
    usage = resp.usage_metadata
    thinking = getattr(usage, 'thoughts_token_count', 0) or 0
    ratio = f"{thinking/budget*100:.0f}%" if budget > 0 else "-"
    print(f"{budget:>6} {thinking:>12} {usage.candidates_token_count:>7} {ratio:>6}")

print("TODO를 완성해주세요")

### 실습 10-4: 비용 계산 함수 구현

토큰 사용량을 기반으로 비용을 계산하는 함수를 구현하세요.

**요구사항**
1. `calculate_cost(usage_metadata)` 함수 구현
2. input, output, thinking 각각의 비용과 합계를 반환
3. 단가: input=$0.15/1M, output=$0.60/1M, thinking=output 단가
4. budget=0과 budget=4096으로 같은 질문을 호출하여 비용 차이를 계산

In [ ]:
# TODO: 비용 계산 함수

# 정답
def calculate_cost(usage_metadata, input_price=0.15/1_000_000, output_price=0.60/1_000_000):
    inp = usage_metadata.prompt_token_count
    out = usage_metadata.candidates_token_count
    think = getattr(usage_metadata, 'thoughts_token_count', 0) or 0

    cost_input = inp * input_price
    cost_output = out * output_price
    cost_thinking = think * output_price
    total = cost_input + cost_output + cost_thinking

    return {
        "input": cost_input,
        "output": cost_output,
        "thinking": cost_thinking,
        "total": total,
        "thinking_ratio": cost_thinking / total * 100 if total > 0 else 0,
    }

question = "퀵소트와 머지소트의 시간복잡도를 비교 분석해줘"

for budget in [0, 4096]:
    resp = client.models.generate_content(
        model=MODEL, contents=question,
        config={"thinking_config": {"thinking_budget": budget}},
    )
    cost = calculate_cost(resp.usage_metadata)
    print(f"budget={budget}: total=${cost['total']:.6f} (thinking 비중: {cost['thinking_ratio']:.1f}%)")

print("TODO를 완성해주세요")

### 실습 10-5: 추론 과정과 답변 품질 비교

추론 과정이 답변 품질에 미치는 영향을 확인하세요.

**요구사항**
1. 논리 퍼즐: "농부가 늑대, 양, 양배추를 강 건너편으로 옮기려고 합니다. 보트에는 농부와 하나의 물건만 탈 수 있습니다. 농부 없이 늑대와 양을 두면 늑대가 양을 먹고, 양과 양배추를 두면 양이 양배추를 먹습니다. 어떻게 해야 할까요?"
2. budget=0과 budget=8192로 각각 호출 (include_thoughts=True)
3. 두 답변의 풀이 단계 수와 정확성을 비교

In [ ]:
# TODO: 추론 과정과 답변 품질 비교

# 정답
puzzle = (
    "농부가 늑대, 양, 양배추를 강 건너편으로 옮기려고 합니다. "
    "보트에는 농부와 하나의 물건만 탈 수 있습니다. "
    "농부 없이 늑대와 양을 두면 늑대가 양을 먹고, "
    "양과 양배추를 두면 양이 양배추를 먹습니다. "
    "어떻게 해야 할까요?"
)

for budget in [0, 8192]:
    resp = client.models.generate_content(
        model=MODEL, contents=puzzle,
        config={"thinking_config": {
            "thinking_budget": budget,
            "include_thoughts": True,
        }},
    )
    thought, answer = extract_thought_and_response(resp)
    usage = resp.usage_metadata
    thinking = getattr(usage, 'thoughts_token_count', 0) or 0

    print(f"=== budget={budget} ===")
    print(f"  추론 과정 길이: {len(thought)}자")
    print(f"  thinking_tokens: {thinking}")
    print(f"  답변:\n  {answer[:300]}")
    print()

print("TODO를 완성해주세요")

### 실습 10-6: LangChain thinking 비교

LangChain의 ChatGoogleGenerativeAI에서 thinking_budget을 제어하세요.

**요구사항**
1. thinking_budget=0과 thinking_budget=2048로 LLM 인스턴스 2개 생성
2. 질문: "파이썬에서 데코레이터의 동작 원리를 설명해줘"
3. 두 결과의 content와 usage_metadata를 비교
4. 어느 쪽이 더 상세한 설명을 제공하는지 확인

In [ ]:
# TODO: LangChain thinking 비교

# 정답
llm_0 = ChatGoogleGenerativeAI(
    model=MODEL, google_api_key=GEMINI_API_KEY, thinking_budget=0,
)
llm_2048 = ChatGoogleGenerativeAI(
    model=MODEL, google_api_key=GEMINI_API_KEY, thinking_budget=2048,
)

question = "파이썬에서 데코레이터의 동작 원리를 설명해줘"

for label, llm_inst in [("budget=0", llm_0), ("budget=2048", llm_2048)]:
    result = llm_inst.invoke(question)
    print(f"=== {label} ===")
    print(f"  답변 길이: {len(result.content)}자")
    print(f"  tokens: {result.usage_metadata}")
    print(f"  답변(일부): {result.content[:150]}")
    print()

print("TODO를 완성해주세요")

### 실습 10-7: Thinking + Structured Output

Thinking 모드와 Structured Output을 함께 사용하세요.

**요구사항**
1. `CodeAnalysis` Pydantic 모델 정의: language(str), bug_count(int), bugs(list[str]), fix_suggestion(str)
2. thinking_budget=4096으로 버그 분석
3. 코드: `def avg(nums): return sum(nums) / len(nums)` (빈 리스트 버그)
4. 구조화된 분석 결과 출력

In [ ]:
# TODO: Thinking + Structured Output

# 정답
class CodeAnalysis(BaseModel):
    language: str = Field(description="프로그래밍 언어")
    bug_count: int = Field(description="발견된 버그 수")
    bugs: list[str] = Field(description="발견된 버그 목록")
    fix_suggestion: str = Field(description="수정 제안 코드")

code = "def avg(nums): return sum(nums) / len(nums)"

resp = client.models.generate_content(
    model=MODEL,
    contents=f"다음 코드의 버그를 분석해줘:\n\n```python\n{code}\n```",
    config={
        "thinking_config": {"thinking_budget": 4096},
        "response_mime_type": "application/json",
        "response_schema": CodeAnalysis,
    },
)

result = CodeAnalysis.model_validate_json(resp.text)
print(f"언어: {result.language}")
print(f"버그 수: {result.bug_count}")
for bug in result.bugs:
    print(f"  - {bug}")
print(f"수정 제안: {result.fix_suggestion}")

usage = resp.usage_metadata
thinking = getattr(usage, 'thoughts_token_count', 0) or 0
print(f"\nthinking_tokens: {thinking}")

print("TODO를 완성해주세요")

---

### 생각해보기

1. 실습 10-3에서 budget을 높여도 실제 thinking_tokens가 budget에 도달하지 않는 경우가 있었나요? 이는 모델이 "충분히 생각했다"고 판단하고 일찍 추론을 종료한 것입니다. 이 현상의 장단점은 무엇일까요?
2. 실습 10-5의 논리 퍼즐에서 budget=0과 budget=8192의 답변 품질 차이는 어느 정도였나요? 모든 논리 문제에서 이 차이가 일관되게 나타날까요?
3. 실습 10-7처럼 코드 분석에 thinking을 사용하면 비용이 증가합니다. 어떤 코드 분석 작업에는 thinking이 필수이고, 어떤 작업에는 불필요할까요?

---

# Part 3 — 챌린지

이론과 실습에서 배운 내용을 응용하여 스스로 설계하고 구현합니다.
정답이 제공되지 않으며, 강사와 함께 진행합니다.

### 챌린지 10-1: 질문 유형별 최적 thinking_budget 탐색 실험 (난이도: 중)

> 이 챌린지는 정답이 제공되지 않습니다. 강사와 함께 진행하세요.

다양한 유형의 질문에 대해 최적의 thinking_budget을 탐색하세요.

**과제**
- 5가지 유형의 질문을 준비 (인사, 번역, 분류, 논리 퍼즐, 수학)
- 각 질문을 budget [0, 512, 2048, 8192]로 테스트
- 각 조합에서: 응답 시간, thinking_tokens, 비용, 답변 품질을 기록
- 결과를 표로 정리하고, 유형별 최적 budget을 결론으로 도출

**힌트**
- 답변 품질은 주관적이지만, 정답이 있는 질문(수학, 논리)은 정확성으로 평가하세요
- 비용 대비 품질 향상이 미미한 지점이 최적 budget입니다
- 결과를 딕셔너리나 리스트에 저장한 뒤 한 번에 출력하면 비교가 쉽습니다

In [ ]:
# 여기에 코드를 작성하세요
# === 챌린지 10-1 답안 ===
import time

# ── 1단계: 질문 + budget 정의 ──
questions = {
    "인사":     "안녕하세요!",
    "번역":     "Translate to English: '오늘 날씨가 참 좋습니다.'",
    "분류":     "다음 텍스트의 감성을 positive/negative/neutral로 분류해줘: '배송이 늦어서 짜증나요'",
    "논리퍼즐": "A는 B보다 크고, C는 A보다 큽니다. B는 D보다 작습니다. 가장 작은 것은?",
    "수학":     "연립방정식을 풀어줘: 2x + y = 10, x - y = 1",
}

budgets = [0, 512, 2048, 8192]

# ── 2단계: 실험 실행 ──
results = []

for q_type, prompt in questions.items():
    for budget in budgets:
        start = time.time()
        resp = client.models.generate_content(
            model=MODEL,
            contents=prompt,
            config={"thinking_config": {"thinking_budget": budget}},
        )
        elapsed = time.time() - start

        usage = resp.usage_metadata
        thinking_tokens = getattr(usage, "thoughts_token_count", 0) or 0
        output_tokens = usage.candidates_token_count

        results.append({
            "유형": q_type,
            "budget": budget,
            "시간": elapsed,
            "thinking": thinking_tokens,
            "출력": output_tokens,
            "답변": resp.text.strip()[:80],
        })

# ── 3단계: 결과 표 출력 ──
print(f"{'유형':<8} {'budget':>7} {'시간(초)':>8} {'think':>7} {'출력':>5}  답변")
print("-" * 90)
for r in results:
    print(f"{r['유형']:<8} {r['budget']:>7} {r['시간']:>8.2f} {r['thinking']:>7} {r['출력']:>5}  {r['답변'][:50]}")

# ── 결론 ──
print("\n결론:")
print("- 인사/번역/분류: budget=0으로 충분 (thinking 토큰 낭비 없이 정확)")
print("- 논리 퍼즐: budget=512~2048에서 정확도 향상, 이후 개선 미미")
print("- 수학: budget=2048 이상에서 풀이 과정이 정교해짐")
print("- 원칙: 단순 작업은 0, 추론 필요 시 512~2048이 비용 대비 최적 구간")

---

### 생각해보기

1. 탐색 결과에서 "비용 대비 품질 향상이 가장 큰 budget 구간"은 어디였나요? 이 구간이 모든 유형에서 동일했나요?
2. 실시간 서비스에서 질문마다 다른 budget을 적용하려면 어떤 시스템 설계가 필요할까요? 라우터, 분류 모델, 캐시 등을 고려해보세요.